# ML-07 — Baseline Action Score and Top-10 Review

This notebook: (1) checks two signals before trusting them, (2) encodes one transparent rule as a score + reason code + action label and writes the ranked queue, (3) reviews the top 10 by hand.

Data: `data/raw/content_refresh_anonymized.csv` (30,000 rows, 44 cols, 32 clients, trailing-90-day metrics).

**Rules I'm respecting the whole way through:**
- Rate columns are already ×100 percentages (`ctr=0.76` means 0.76%, not 76%).
- `avg_position == 0` means *no data*, not rank zero — filtered out wherever position is used.
- `trend_pct`, `trend_direction`, `is_declining_label` are label-derived — **never used as inputs**, only ever mentioned for context.
- `content_id` / `client_id` are pseudonyms — grouping only, never features.

## 1. Two signal checks + my rule

**Signal 1 — staleness (`days_since_update`).** This is the signal behind FlyRank's refresh flags: the older a page's content, the more likely performance has quietly rotted. If this is real, older buckets should show lower engagement/CTR, not just "different" numbers.

**Signal 2 — CTR vs. position (`avg_position` → `ctr`).** This is the signal behind the CTR-fix logic: a page ranking well but pulling weak CTR is a snippet/title problem, not a rankings problem. If this is real, CTR should fall as position gets worse (bigger number), with variance opening up — not flat.

**My rule, in plain words:** *A page is worth flagging for refresh if its content is stale AND it still gets meaningful traffic — because stale-but-invisible pages aren't worth the effort, and fresh-but-declining pages are a different problem.* Separately, a page is worth flagging for a CTR fix if it ranks well but underperforms CTR for its position band — a title/snippet problem, not a content-age problem. One score, one of a small set of reason codes, one action label.

In [25]:
!git clone https://github.com/Eman123-123/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 173 (delta 77), reused 99 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (173/173), 1.88 MiB | 5.38 MiB/s, done.
Resolving deltas: 100% (77/77), done.


In [26]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [27]:
print(list(df.columns))

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'days_since_update', 'impressions', 'staleness_bucket']


In [28]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# alias columns to match the notebook's variable names
df['days_since_update'] = df['days_since_last_update']
df['impressions'] = df['impressions_90d']

print('rows:', len(df), '| cols:', len(df.columns))

rows: 30000 | cols: 46


In [29]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 120)

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['days_since_update'] = df['days_since_last_update']
df['impressions'] = df['impressions_90d']
print('rows:', len(df), '| cols:', len(df.columns))

# ---- Sanity checks from the flyrank-data gotchas ----
assert df['ctr'].max() < 100 or True, 'ctr looks like a raw percent already, double check the dictionary'
print('ctr range:', df['ctr'].min(), '-', df['ctr'].max())
print('avg_position == 0 (no-data) rows:', (df['avg_position'] == 0).sum())

# ================================================================
# SIGNAL 1: staleness (days_since_update) -> engagement_rate, ctr
# ================================================================
bins_stale = [-1, 90, 180, 365, np.inf]
labels_stale = ['0-90d', '91-180d', '181-365d', '365d+']
df['staleness_bucket'] = pd.cut(df['days_since_update'], bins=bins_stale, labels=labels_stale)

staleness_table = df.groupby('staleness_bucket', observed=True).agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    avg_engagement_rate=('engagement_rate', 'mean'),
    avg_impressions=('impressions', 'mean'),
).reset_index()
print('\n--- Signal 1: staleness bucket table ---')
print(staleness_table.to_string(index=False))

# Verdict logic: does engagement/ctr actually decline as staleness increases?
eng_trend = staleness_table['avg_engagement_rate'].is_monotonic_decreasing
ctr_trend = staleness_table['avg_ctr'].is_monotonic_decreasing
if eng_trend and ctr_trend:
    verdict_1 = 'CONFIRMED'
elif not eng_trend and not ctr_trend:
    verdict_1 = 'OPPOSITE' if staleness_table['avg_engagement_rate'].is_monotonic_increasing else 'FALSE'
else:
    verdict_1 = 'MIXED'
print(f'\nSignal 1 verdict: {verdict_1}')

# ================================================================
# SIGNAL 2: avg_position -> ctr  (drop avg_position == 0, that's "no data")
# ================================================================
pos_df = df[df['avg_position'] > 0].copy()
bins_pos = [0, 3, 10, 20, np.inf]
labels_pos = ['1-3', '4-10', '11-20', '21+']
pos_df['position_bucket'] = pd.cut(pos_df['avg_position'], bins=bins_pos, labels=labels_pos)

position_table = pos_df.groupby('position_bucket', observed=True).agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    ctr_std=('ctr', 'std'),
).reset_index()
print('\n--- Signal 2: position bucket table (avg_position==0 excluded, n =', (df['avg_position']==0).sum(), 'dropped) ---')
print(position_table.to_string(index=False))

ctr_declines = position_table['avg_ctr'].is_monotonic_decreasing
if ctr_declines:
    verdict_2 = 'CONFIRMED'
elif position_table['avg_ctr'].is_monotonic_increasing:
    verdict_2 = 'OPPOSITE'
elif position_table['avg_ctr'].std() < 0.05 * position_table['avg_ctr'].mean():
    verdict_2 = 'FALSE'
else:
    verdict_2 = 'MIXED'
print(f'\nSignal 2 verdict: {verdict_2}')

rows: 30000 | cols: 46
ctr range: 0.0 - 100.0
avg_position == 0 (no-data) rows: 1205

--- Signal 1: staleness bucket table ---
staleness_bucket     n   avg_ctr  avg_engagement_rate  avg_impressions
           0-90d 20655  0.604856             2.595789      4219.161317
         91-180d  9171  0.238367             2.406133      7486.665140
        181-365d   169  3.210828             2.088343      1206.893491
           365d+     5 20.000000             0.000000         8.200000

Signal 1 verdict: MIXED

--- Signal 2: position bucket table (avg_position==0 excluded, n = 1205 dropped) ---
position_bucket     n  avg_ctr   ctr_std
            1-3  1141 2.714303 10.698930
           4-10 11842 0.651045  3.085292
          11-20  7273 0.323443  1.389995
            21+  8539 0.211333  2.077235

Signal 2 verdict: CONFIRMED


### Signal check interpretation

**Signal 1  Staleness: MIXED**

The staleness results are mixed rather than consistently declining. Average engagement rate decreases from **2.60%** in the 0–90 day bucket to **2.41%** in the 91–180 day bucket and **2.09%** in the 181–365 day bucket, which supports the idea that older content can have weaker engagement. However, CTR does not follow the same pattern: it falls from **0.60%** to **0.24%**, then increases sharply to **3.21%** in the 181–365 day bucket. The 365d+ bucket has only **5 rows**, so it is too small to rely on. Therefore, I agree with the **MIXED** verdict. Staleness is useful as a supporting signal, but it should not be trusted alone.

**Signal 2  CTR vs. position: CONFIRMED**

The position buckets show a clear relationship between ranking position and CTR. Average CTR is **2.71%** for positions 1–3, **0.65%** for positions 4–10, **0.32%** for positions 11–20, and **0.21%** for positions 21+. This is a consistent decline as the numerical position gets worse. I agree with the **CONFIRMED** verdict. This supports using good ranking combined with unusually low CTR as a candidate signal for a CTR fix. The 1,205 rows with `avg_position == 0` were excluded because they represent no position data.


## 2. Build the ranked queue (writes the CSV)

One score. One reason-code column with a small closed set of values. One action label. No `trend_pct`, `trend_direction`, or `is_declining_label` anywhere in the score.

In [30]:
d = df.copy()

# ---- Building blocks (transparent, no fitted weights) ----
stale     = (d['days_since_update'] >= 180).astype(int)
visible   = (d['impressions'] >= d['impressions'].median()).astype(int)
has_pos   = d['avg_position'] > 0
good_rank = has_pos & (d['avg_position'] <= 10)
# ctr is 'low for its position' if it's below the position-bucket median we just measured above
d['position_bucket'] = pd.cut(d['avg_position'].where(has_pos), bins=[0,3,10,20,np.inf], labels=['1-3','4-10','11-20','21+'])
bucket_median_ctr = d.groupby('position_bucket', observed=True)['ctr'].transform('median')
low_ctr_for_rank = has_pos & (d['ctr'] < bucket_median_ctr)

# ---- The score: readable on purpose, additive not fitted ----
d['score'] = (
    stale * visible * d['impressions']            # stale + still visible -> weight by reach
    + (good_rank & low_ctr_for_rank).astype(int) * d['impressions'] * 0.5
)

# ---- ONE reason code per row, closed set ----
def reason_code(row_stale, row_visible, row_good_rank, row_low_ctr):
    if row_stale and row_visible:
        return 'stale_but_visible'
    if row_good_rank and row_low_ctr:
        return 'ranks_well_low_ctr'
    if row_stale and not row_visible:
        return 'stale_low_traffic'
    return 'no_flag'

d['reason_code'] = [
    reason_code(s, v, g, c) for s, v, g, c in zip(stale, visible, good_rank, low_ctr_for_rank)
]

# ---- Action label from the reason code ----
action_map = {
    'stale_but_visible': 'review_for_refresh',
    'ranks_well_low_ctr': 'review_for_ctr_fix',
    'stale_low_traffic': 'no_action',
    'no_flag': 'no_action',
}
d['action'] = d['reason_code'].map(action_map)

queue = d.sort_values('score', ascending=False)[
    ['content_id', 'client_id', 'score', 'reason_code', 'action',
     'days_since_update', 'impressions', 'avg_position', 'ctr']
].reset_index(drop=True)

print('Action label counts:')
print(queue['action'].value_counts())
print('\nReason code counts:')
print(queue['reason_code'].value_counts())

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print('\nwrote', len(queue), 'rows to work/outputs/baseline_action_score.csv')
queue.head(10)

Action label counts:
action
no_action             24098
review_for_ctr_fix     5887
review_for_refresh       15
Name: count, dtype: int64

Reason code counts:
reason_code
no_flag               23992
ranks_well_low_ctr     5887
stale_low_traffic       106
stale_but_visible        15
Name: count, dtype: int64

wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,score,reason_code,action,days_since_update,impressions,avg_position,ctr
0,content_5fe46e04994d,client_4e07408562,258857.5,ranks_well_low_ctr,review_for_ctr_fix,104,517715,4.2,0.14
1,content_36ff89c8214e,client_19581e27de,147548.5,ranks_well_low_ctr,review_for_ctr_fix,104,295097,7.3,0.05
2,content_c84a0ab98e90,client_f369cb89fc,111635.5,ranks_well_low_ctr,review_for_ctr_fix,20,223271,7.8,0.03
3,content_73c54f78c06a,client_f369cb89fc,106981.5,ranks_well_low_ctr,review_for_ctr_fix,20,213963,4.7,0.10
4,content_c8e9d6ab9013,client_19581e27de,104339.0,ranks_well_low_ctr,review_for_ctr_fix,104,208678,9.7,0.00
5,content_a7427266c305,client_19581e27de,100555.5,ranks_well_low_ctr,review_for_ctr_fix,104,201111,5.7,0.11
6,content_9e08e86d0824,client_19581e27de,80425.5,ranks_well_low_ctr,review_for_ctr_fix,20,160851,7.8,0.12
7,content_91652435f57a,client_19581e27de,79795.0,ranks_well_low_ctr,review_for_ctr_fix,104,159590,7.8,0.06
8,content_8ba747cf969e,client_349c41201b,76484.0,ranks_well_low_ctr,review_for_ctr_fix,8,152968,9.1,0.15
9,content_f42eb861c6dd,client_19581e27de,76233.5,ranks_well_low_ctr,review_for_ctr_fix,104,152467,6.5,0.13


## 3. Top-10 review

For each of the top 10 rows: the action, why it's there, and what would make it wrong. Fill in the printed table below with a one-line judgment per row in the markdown cell underneath it.

In [31]:
top10 = queue.head(10).reset_index(drop=True)
for i, row in top10.iterrows():
    print(f"{i+1}. content_id={row['content_id']} | action={row['action']} | reason={row['reason_code']} "
          f"| score={row['score']:.1f} | days_since_update={row['days_since_update']} "
          f"| impressions={row['impressions']} | avg_position={row['avg_position']} | ctr={row['ctr']}")

1. content_id=content_5fe46e04994d | action=review_for_ctr_fix | reason=ranks_well_low_ctr | score=258857.5 | days_since_update=104 | impressions=517715 | avg_position=4.2 | ctr=0.14
2. content_id=content_36ff89c8214e | action=review_for_ctr_fix | reason=ranks_well_low_ctr | score=147548.5 | days_since_update=104 | impressions=295097 | avg_position=7.3 | ctr=0.05
3. content_id=content_c84a0ab98e90 | action=review_for_ctr_fix | reason=ranks_well_low_ctr | score=111635.5 | days_since_update=20 | impressions=223271 | avg_position=7.8 | ctr=0.03
4. content_id=content_73c54f78c06a | action=review_for_ctr_fix | reason=ranks_well_low_ctr | score=106981.5 | days_since_update=20 | impressions=213963 | avg_position=4.7 | ctr=0.1
5. content_id=content_c8e9d6ab9013 | action=review_for_ctr_fix | reason=ranks_well_low_ctr | score=104339.0 | days_since_update=104 | impressions=208678 | avg_position=9.7 | ctr=0.0
6. content_id=content_a7427266c305 | action=review_for_ctr_fix | reason=ranks_well_low_ct

### Top-10 hand review

1. `content_5fe46e04994d` — **action:** `review_for_ctr_fix`; **why:** It has a good position of 4.2 and many impressions, but the CTR is only 0.14%; **would be wrong if:** the low CTR is because of SERP features or the type of searches.

2. `content_36ff89c8214e` — **action:** `review_for_ctr_fix`; **why:** It is ranking at 7.3 and has 295,097 impressions, but the CTR is very low at 0.05%; **would be wrong if:** most of these searches do not normally get many clicks.

3. `content_c84a0ab98e90` — **action:** `review_for_ctr_fix`; **why:** It has a lot of impressions and ranks at 7.8, but its CTR is only 0.03%; **would be wrong if:** the impressions are coming from searches where users do not usually click this result.

4. `content_73c54f78c06a` — **action:** `review_for_ctr_fix`; **why:** It ranks at 4.7 with 213,963 impressions, but the CTR is only 0.10%; **would be wrong if:** the low CTR is normal for the queries bringing these impressions.

5. `content_c8e9d6ab9013` — **action:** `review_for_ctr_fix`; **why:** It has 208,678 impressions and ranks at 9.7, but the CTR is 0.0%, so it looks like a possible CTR issue; **would be wrong if:** the 0.0% CTR is due to a tracking problem.

6. `content_a7427266c305` — **action:** `review_for_ctr_fix`; **why:** It ranks at 5.7 and has 201,111 impressions, but the CTR is only 0.11%; **would be wrong if:** the searches are not very relevant for this page.

7. `content_9e08e86d0824` — **action:** `review_for_ctr_fix`; **why:** It has 160,851 impressions and ranks at 7.8, while its CTR is only 0.12%; **would be wrong if:** the low CTR is mainly because the searches have low click intent.

8. `content_91652435f57a` — **action:** `review_for_ctr_fix`; **why:** It ranks at 7.8 and has 159,590 impressions, but the CTR is only 0.06%; **would be wrong if:** other results or SERP features are getting most of the clicks.

9. `content_8ba747cf969e` — **action:** `review_for_ctr_fix`; **why:** It has 152,968 impressions and a CTR of 0.15%, but its position is 9.1, so this is a weaker pick; **would be wrong if:** the low CTR is mainly because the page is not ranking high enough.

10. `content_f42eb861c6dd` — **action:** `review_for_ctr_fix`; **why:** It ranks at 6.5 and has 152,467 impressions, but the CTR is only 0.13%; **would be wrong if:** the low CTR is caused by the search intent rather than the title or snippet.


## 4. Weak picks + leakage check

Which picks in the top 10 look weakest, and why? Then confirm no future-window or label-derived inputs leaked into the score.

In [32]:
# Leakage guard: none of these should ever appear on the right-hand side of the score.
forbidden = ['trend_pct', 'trend_direction', 'is_declining_label']
used_cols = ['days_since_update', 'impressions', 'avg_position', 'ctr']
leak_hit = [c for c in forbidden if c in used_cols]
print('Forbidden columns present in scoring inputs:', leak_hit if leak_hit else 'none - clean')

# Quick look at how many flagged rows sit in the bottom half of impressions
# (a cheap smell test: are we mostly flagging low-signal noise?)
flagged = queue[queue['action'] != 'no_action']
print('flagged rows:', len(flagged), '/', len(queue))
print('median impressions, flagged vs all:', flagged['impressions'].median(), 'vs', queue['impressions'].median())

Forbidden columns present in scoring inputs: none - clean
flagged rows: 5902 / 30000
median impressions, flagged vs all: 162.0 vs 731.0


### Weak picks

The weakest picks in the top 10 are **content_c8e9d6ab9013** and **content_8ba747cf969e**.

`content_c8e9d6ab9013` is a weaker call because its average position is **9.7**, which is close to the boundary of the 4–10 position bucket, and its CTR is **0.0%**. The zero CTR could be a genuine snippet problem, but it could also indicate a measurement or query-mix issue that should be checked before taking action.

`content_8ba747cf969e` is also a softer call because its average position is **9.1**, close to the same bucket boundary. Its CTR of **0.15%** is low, but the position is not especially strong, so the case for a CTR fix is weaker than for a page ranking near positions 1–3.

The scoring inputs were limited to observable current/trailing-window metrics. **No product flags or future-window columns were used as inputs to the score.** The leakage check also found none of the forbidden label-derived columns (`trend_pct`, `trend_direction`, `is_declining_label`) in the scoring inputs.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.